In [ ]:
# Install all required packages
!pip install -q \
    groq \
    langchain \
    langchain-groq \
    langgraph \
    langchain-community \
    qdrant-client \
    sentence-transformers \
    arxiv \
    duckduckgo-search \
    pymupdf \
    streamlit \
    pyngrok \
    pydantic \
    python-dotenv

print("Dependencies installed.")

In [ ]:
from google.colab import userdata
import os

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

print(f"API key loaded — length: {len(GROQ_API_KEY)} characters")

In [ ]:
from groq import Groq

client = Groq()

# Using Llama 3.3 70B — strong open-source model, free on Groq
MODEL = "llama-3.3-70b-versatile"

# Verify the connection works
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user",   "content": "What is machine learning in one sentence?"}
    ],
    temperature=0.7,
    max_tokens=100
)

print("Model response:", response.choices[0].message.content)
print("Tokens used:", response.usage.total_tokens)

In [ ]:
from typing import Optional

def call_llm(
    user_message: str,
    system_prompt: str,
    temperature: float = 0.7,
    max_tokens: int = 1500
) -> str:
    """
    Wrapper around the Groq chat completion API.
    All agents in this system use this function as their LLM interface.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


# Sanity check
result = call_llm(
    user_message="What is retrieval-augmented generation?",
    system_prompt="Answer in two sentences."
)
print(result)

In [ ]:
PLANNER_SYSTEM_PROMPT = """
You are the Planner Agent in a multi-agent research system.

Your job is to analyze a research question and produce a structured research plan.

Respond in this exact format:

## RESEARCH QUESTION ANALYSIS
**Type:** [Empirical / Theoretical / Survey / Applied]
**Complexity:** [Low / Medium / High]
**Domain:** [e.g., Machine Learning, NLP, Computer Vision]

## SUB-QUESTIONS TO INVESTIGATE
1. [Sub-question]
2. [Sub-question]
3. [Sub-question]
4. [Sub-question]
5. [Sub-question]

## SEARCH STRATEGY
**Keywords for paper search:** [comma-separated keywords]
**Arxiv categories:** [e.g., cs.CL, cs.LG, cs.CV]
**What to look for:** [2-3 sentences describing relevant sources]

## EXPECTED OUTPUT
**Report sections:** [list the sections the final report should have]

Think like a PhD researcher. Do not answer the question — only plan how to research it.
"""


def planner_agent(research_question: str) -> dict:
    """
    Decomposes a research question into a structured plan.
    Returns a dict with 'question' and 'plan' keys.
    """
    print(f"Planner: processing '{research_question[:60]}...'")

    plan = call_llm(
        user_message=f"Research Question: {research_question}",
        system_prompt=PLANNER_SYSTEM_PROMPT,
        temperature=0.3,
        max_tokens=1000
    )

    return {"question": research_question, "plan": plan}


print("Planner agent ready.")

In [ ]:
research_question = "How do large language models handle long-context reasoning compared to traditional transformers?"

result = planner_agent(research_question)
print(result["plan"])

In [ ]:
import arxiv
import time
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
from typing import List, Dict


def arxiv_search(keywords: str, max_results: int = 5) -> List[Dict]:
    """
    Searches Arxiv for research papers matching the given keywords.
    Uses the arxiv library with exponential backoff, falling back to
    direct HTTP if the library is rate-limited.
    """
    print(f"Arxiv search: '{keywords}'")

    # Attempt 1: use arxiv library with retries
    for attempt in range(3):
        try:
            if attempt > 0:
                wait = [5, 15, 30][attempt]
                print(f"  Retry {attempt}/2 — waiting {wait}s")
                time.sleep(wait)

            arxiv_client = arxiv.Client(
                page_size=max_results,
                delay_seconds=5,
                num_retries=2
            )
            search = arxiv.Search(
                query=keywords,
                max_results=max_results,
                sort_by=arxiv.SortCriterion.Relevance
            )
            papers = []
            for paper in arxiv_client.results(search):
                papers.append({
                    "title":      paper.title,
                    "authors":    ", ".join(a.name for a in paper.authors[:3]),
                    "abstract":   paper.summary[:800],
                    "url":        paper.entry_id,
                    "published":  str(paper.published.date()),
                    "categories": ", ".join(paper.categories)
                })
            if papers:
                print(f"  Found {len(papers)} papers")
                return papers
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {type(e).__name__}")

    # Attempt 2: direct HTTP request to Arxiv API
    print("  Falling back to direct HTTP request")
    time.sleep(10)
    try:
        encoded = urllib.parse.quote(keywords)
        url = (
            f"http://export.arxiv.org/api/query"
            f"?search_query=all:{encoded}&start=0&max_results={max_results}&sortBy=relevance"
        )
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            content = resp.read()

        ns = {"atom": "http://www.w3.org/2005/Atom"}
        root = ET.fromstring(content)
        papers = []

        for entry in root.findall("atom:entry", ns):
            title     = entry.find("atom:title", ns)
            summary   = entry.find("atom:summary", ns)
            link      = entry.find("atom:id", ns)
            published = entry.find("atom:published", ns)
            authors   = entry.findall("atom:author", ns)
            names = []
            for a in authors[:3]:
                name = a.find("atom:name", ns)
                if name is not None:
                    names.append(name.text)

            papers.append({
                "title":      title.text.strip()         if title     is not None else "Unknown",
                "authors":    ", ".join(names),
                "abstract":   summary.text.strip()[:800] if summary   is not None else "",
                "url":        link.text.strip()           if link      is not None else "",
                "published":  published.text[:10]         if published is not None else "",
                "categories": ""
            })

        print(f"  Found {len(papers)} papers via direct HTTP")
        return papers

    except Exception as e:
        print(f"  HTTP fallback failed: {e}")

    # Fallback: return a known paper so the pipeline does not break
    print("  Using fallback paper data")
    return [
        {
            "title":     "Attention Is All You Need",
            "authors":   "Vaswani, A., Shazeer, N., Parmar, N.",
            "abstract":  "The dominant sequence transduction models are based on complex recurrent or "
                         "convolutional neural networks. We propose the Transformer, based solely on "
                         "attention mechanisms, dispensing with recurrence and convolutions entirely.",
            "url":       "https://arxiv.org/abs/1706.03762",
            "published": "2017-06-12",
            "categories": "cs.CL"
        },
        {
            "title":     "Longformer: The Long-Document Transformer",
            "authors":   "Beltagy, I., Peters, M., Cohan, A.",
            "abstract":  "Transformer-based models are unable to process long sequences due to their "
                         "self-attention operation, which scales quadratically with sequence length. "
                         "We introduce the Longformer with an attention mechanism that scales linearly.",
            "url":       "https://arxiv.org/abs/2004.05150",
            "published": "2020-04-10",
            "categories": "cs.CL"
        }
    ]


# Test
test_papers = arxiv_search("large language models long context", max_results=3)
for i, p in enumerate(test_papers, 1):
    print(f"{i}. {p['title']} ({p['published']})")

In [ ]:
from duckduckgo_search import DDGS


def web_search(query: str, max_results: int = 5) -> List[Dict]:
    """
    Searches the web using DuckDuckGo. No API key required.
    Returns a list of results with title, url, and snippet.
    """
    print(f"Web search: '{query}'")
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append({
                    "title":   r.get("title", ""),
                    "url":     r.get("href", ""),
                    "snippet": r.get("body", "")[:600]
                })
        print(f"  Found {len(results)} results")
    except Exception as e:
        print(f"  Web search failed: {e} (Arxiv results will still be used)")
    return results


# Test
test_web = web_search("LLM long context reasoning", max_results=3)
for i, r in enumerate(test_web, 1):
    print(f"{i}. {r['title']}")

In [ ]:
import json

RESEARCHER_SYSTEM_PROMPT = """
You are the Research Agent in a multi-agent research system.

You receive a research plan and search results (papers and web).
Extract and organize the most relevant information into structured findings.

Respond in this exact format:

## KEY FINDINGS

### Finding 1: [Short descriptive title]
**Source:** [Paper title or website name]
**Evidence:** [Specific data, results, or claims from the source]
**Relevance:** [Why this matters to the research question]

[Repeat for each significant finding — minimum 3, maximum 8]

## RESEARCH GAPS
[2-3 sentences on what the sources do not cover]

## CONFIDENCE LEVEL
**Overall:** [Low / Medium / High]
**Reason:** [One sentence explanation]

Base all findings on the provided sources. Do not fabricate citations.
"""


def research_agent(plan: dict, max_papers: int = 4, max_web: int = 3) -> dict:
    """
    Searches Arxiv and the web based on the plan, then synthesizes findings.
    Returns a dict with papers, web_results, and findings.
    """
    print("Research agent starting...")

    # Extract search keywords from the plan
    keyword_prompt = (
        "From this research plan, extract the search keywords. "
        "Return them as a single search query string, max 8 words.\n\n"
        f"Plan:\n{plan['plan']}"
    )
    keywords = call_llm(
        user_message=keyword_prompt,
        system_prompt="Return only the search keywords, nothing else.",
        temperature=0.1,
        max_tokens=50
    ).strip()

    print(f"  Keywords: {keywords}")

    papers      = arxiv_search(keywords, max_results=max_papers)
    web_results = web_search(keywords, max_results=max_web)

    # Build context for synthesis
    context = f"RESEARCH QUESTION: {plan['question']}\n\n=== ARXIV PAPERS ===\n"
    for i, p in enumerate(papers, 1):
        context += (
            f"\nPaper {i}: {p['title']}\n"
            f"Authors: {p['authors']} | Published: {p['published']}\n"
            f"Abstract: {p['abstract']}\nURL: {p['url']}\n---"
        )

    context += "\n\n=== WEB RESULTS ===\n"
    for i, r in enumerate(web_results, 1):
        context += (
            f"\nResult {i}: {r['title']}\n"
            f"URL: {r['url']}\nContent: {r['snippet']}\n---"
        )

    print("  Synthesizing findings...")
    findings = call_llm(
        user_message=context,
        system_prompt=RESEARCHER_SYSTEM_PROMPT,
        temperature=0.3,
        max_tokens=1500
    )

    return {
        "question":    plan["question"],
        "plan":        plan["plan"],
        "papers":      papers,
        "web_results": web_results,
        "findings":    findings
    }


print("Research agent ready.")

In [ ]:
research_question = "How do large language models handle long-context reasoning compared to traditional transformers?"

plan     = planner_agent(research_question)
research = research_agent(plan)

print(f"\nPapers retrieved: {len(research['papers'])}")
for i, p in enumerate(research['papers'], 1):
    print(f"  {i}. {p['title']} ({p['published']})")

print("\n" + "="*60)
print(research['findings'])

In [ ]:
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, END


class ResearchState(TypedDict):
    """
    Shared state object that flows through every agent in the graph.
    Each agent reads only the fields it needs and writes only the fields it produces.
    """
    question:             str        # original research question
    plan:                 str        # output of the planner
    papers:               List[Dict] # raw papers from Arxiv
    web_results:          List[Dict] # raw web results
    findings:             str        # synthesized findings from researcher
    summary:              str        # compressed summary
    verified:             str        # citation verification report
    verification_passed:  bool       # whether verification passed
    report:               str        # final technical report


print("ResearchState fields:", list(ResearchState.__annotations__.keys()))

In [ ]:
SUMMARIZER_SYSTEM_PROMPT = """
You are the Summarizer Agent in a multi-agent research system.

Compress the raw findings into a concise, information-dense summary.

Rules:
- Maximum 400 words
- Retain all specific numbers, results, and technical terms
- Remove repetition
- Use bullet points for key findings
- Preserve source attribution
- End with 2-3 sentences on what remains unknown

Format:

## CORE ANSWER
[2-3 sentences directly answering the research question]

## KEY FINDINGS
- [Finding with source]
- [Finding with source]
...

## OPEN QUESTIONS
[What remains unknown or contested]
"""


def summarizer_node(state: ResearchState) -> dict:
    """
    Compresses research findings into a structured summary.
    Reads: findings, question
    Writes: summary
    """
    print("Summarizer running...")

    prompt = f"Research Question: {state['question']}\n\nFindings:\n{state['findings']}"

    summary = call_llm(
        user_message=prompt,
        system_prompt=SUMMARIZER_SYSTEM_PROMPT,
        temperature=0.2,
        max_tokens=600
    )

    print("  Summary complete.")
    return {"summary": summary}


print("Summarizer agent ready.")

In [ ]:
VERIFIER_SYSTEM_PROMPT = """
You are the Citation Verifier in a multi-agent research system.
Your job is to reduce hallucination by checking factual claims against real sources.

For each factual claim in the summary, assign one of:
- VERIFIED: directly supported by a provided source
- INFERRED: reasonable inference from sources, not directly stated
- UNVERIFIED: no support found in provided sources

Output format:

## VERIFICATION REPORT

### Claim 1
**Claim:** [the claim]
**Status:** VERIFIED / INFERRED / UNVERIFIED
**Source:** [supporting source, or 'None found']

[Repeat for each major claim]

## OVERALL ASSESSMENT
**Verified claims:** X/Y
**Trust score:** High (>80%) / Medium (50-80%) / Low (<50%)
**Recommendation:** Safe to use / Use with caution / Needs more research
"""


def citation_verifier_node(state: ResearchState) -> dict:
    """
    Cross-checks claims in the summary against the fetched sources.
    Reads: summary, papers, web_results
    Writes: verified, verification_passed
    """
    print("Citation verifier running...")

    sources = "\n=== SOURCES ===\n"
    for i, p in enumerate(state["papers"], 1):
        sources += f"\n[Paper {i}] {p['title']} by {p['authors']}\nAbstract: {p['abstract'][:400]}\n"
    for i, r in enumerate(state.get("web_results", []), 1):
        sources += f"\n[Web {i}] {r['title']}\nContent: {r['snippet'][:300]}\n"

    prompt = f"Summary to verify:\n{state['summary']}\n\n{sources}"

    verified = call_llm(
        user_message=prompt,
        system_prompt=VERIFIER_SYSTEM_PROMPT,
        temperature=0.1,
        max_tokens=1200
    )

    verification_passed = "Low" not in verified and "UNVERIFIED" not in verified
    print(f"  Verification complete. Passed: {verification_passed}")

    return {"verified": verified, "verification_passed": verification_passed}


print("Citation verifier ready.")

In [ ]:
def planner_node(state: ResearchState) -> dict:
    """LangGraph node: runs the planner agent."""
    result = planner_agent(state["question"])
    return {"plan": result["plan"]}


def researcher_node(state: ResearchState) -> dict:
    """LangGraph node: runs the research agent."""
    result = research_agent({
        "question": state["question"],
        "plan":     state["plan"]
    })
    return {
        "papers":      result["papers"],
        "web_results": result["web_results"],
        "findings":    result["findings"]
    }


print("LangGraph node wrappers defined.")

In [ ]:
REPORT_WRITER_SYSTEM_PROMPT = """
You are the Report Writer in a multi-agent research system.
Write a complete, publication-ready technical report in Markdown.

Structure:

# [Report Title]

## Abstract
[150-200 word summary]

## 1. Introduction
[Context, motivation, scope]

## 2. Background
[Key concepts needed to understand the findings]

## 3. Key Findings
[Detailed findings with source attribution]

## 4. Analysis
[Synthesis, patterns, contradictions across findings]

## 5. Limitations
[What the research could not cover; verification gaps]

## 6. Conclusion
[Direct answer to the research question; future directions]

## References
[All sources used]

Rules:
- Use academic tone
- Only include VERIFIED or INFERRED claims
- Cite sources for every major claim
"""


def report_writer_node(state: ResearchState) -> dict:
    """
    Writes the final technical report from the verified summary.
    Reads: question, plan, summary, verified, papers
    Writes: report
    """
    print("Report writer running...")

    references = "\n".join([
        f"- {p['title']} — {p['authors']} ({p['published']}) — {p['url']}"
        for p in state["papers"]
    ])

    prompt = (
        f"Research Question: {state['question']}\n\n"
        f"Research Plan:\n{state['plan'][:800]}\n\n"
        f"Verified Summary:\n{state['summary']}\n\n"
        f"Verification Report:\n{state['verified'][:600]}\n\n"
        f"References:\n{references}"
    )

    report = call_llm(
        user_message=prompt,
        system_prompt=REPORT_WRITER_SYSTEM_PROMPT,
        temperature=0.4,
        max_tokens=2000
    )

    print("  Report complete.")
    return {"report": report}


print("Report writer agent ready.")

In [ ]:
def build_research_graph():
    """
    Compiles the 5-agent research graph.
    Flow: planner -> researcher -> summarizer -> citation_verifier -> report_writer
    """
    graph = StateGraph(ResearchState)

    graph.add_node("planner",           planner_node)
    graph.add_node("researcher",        researcher_node)
    graph.add_node("summarizer",        summarizer_node)
    graph.add_node("citation_verifier", citation_verifier_node)
    graph.add_node("report_writer",     report_writer_node)

    graph.set_entry_point("planner")
    graph.add_edge("planner",           "researcher")
    graph.add_edge("researcher",        "summarizer")
    graph.add_edge("summarizer",        "citation_verifier")
    graph.add_edge("citation_verifier", "report_writer")
    graph.add_edge("report_writer",     END)

    return graph.compile()


research_graph = build_research_graph()
print("Research graph compiled.")

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
import uuid

print("Loading sentence transformer model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
VECTOR_SIZE = 384
print("Model loaded.")

qdrant = QdrantClient(":memory:")
COLLECTION = "research_memory"

try:
    qdrant.recreate_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
    )
except Exception:
    qdrant.delete_collection(COLLECTION)
    qdrant.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
    )

print(f"Qdrant collection '{COLLECTION}' ready.")


def embed(text: str) -> list:
    """Convert text to a 384-dimensional embedding vector."""
    return embedding_model.encode(text).tolist()


def save_to_memory(question: str, summary: str, report: str) -> str:
    """Store a research result in the vector database."""
    point_id = str(uuid.uuid4())
    qdrant.upsert(
        collection_name=COLLECTION,
        points=[PointStruct(
            id=point_id,
            vector=embed(question),
            payload={"question": question, "summary": summary, "report": report[:2000]}
        )]
    )
    print(f"Saved to memory: '{question[:70]}'")
    return point_id


def search_memory(question: str, threshold: float = 0.75):
    """
    Search for a previously researched similar question.
    Returns the cached result if similarity exceeds the threshold, else None.
    """
    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=embed(question),
        limit=1
    ).points

    if results and results[0].score >= threshold:
        print(f"Memory hit — similarity: {results[0].score:.3f}")
        return results[0].payload

    best = results[0].score if results else 0.0
    print(f"No memory match (best score: {best:.3f})")
    return None


# Verify embedding works
test_vec = embed("test")
print(f"Embedding dimensions: {len(test_vec)}")

In [ ]:
def run_research(question: str) -> dict:
    """
    Main entry point for the research system.
    Checks memory first; runs the full agent pipeline only if no cached result exists.
    Saves new results to memory automatically.
    """
    print(f"\nQuestion: {question}")
    print("-" * 60)

    cached = search_memory(question)
    if cached:
        print("Retrieved from memory.")
        return {
            "question":    question,
            "summary":     cached["summary"],
            "report":      cached["report"],
            "from_memory": True
        }

    print("Running agent pipeline...")
    graph = build_research_graph()

    state = graph.invoke({
        "question":            question,
        "plan":                "",
        "papers":              [],
        "web_results":         [],
        "findings":            "",
        "summary":             "",
        "verified":            "",
        "verification_passed": False,
        "report":              ""
    })

    save_to_memory(question, state["summary"], state["report"])
    state["from_memory"] = False
    return state


print("run_research() ready.")

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/MultiAgentResearch"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Reports will be saved to: {SAVE_DIR}")


def save_report(question: str, report: str) -> str:
    """Save the report as a Markdown file to Google Drive."""
    filename = question[:50].replace(" ", "_").replace("?", "").replace("/", "_")
    filepath = f"{SAVE_DIR}/{filename}.md"
    with open(filepath, "w") as f:
        f.write(f"# Research Report\n**Question:** {question}\n\n---\n\n{report}")
    print(f"Report saved: {filepath}")
    return filepath


question = "How do large language models handle long-context reasoning compared to traditional transformers?"
result   = run_research(question)

print("\n" + "="*60)
print("FINAL REPORT")
print("="*60)
print(result["report"])

source = "memory cache" if result.get("from_memory") else "live research"
print(f"\nSource: {source}")
save_report(question, result["report"])

In [ ]:
# Test with a similar question — should retrieve from memory
similar = "What techniques do LLMs use for reasoning over long contexts?"
result2 = run_research(similar)

if result2.get("from_memory"):
    print("Retrieved from memory — no web search needed.")
else:
    print("Memory threshold not met — ran fresh pipeline.")
    print("Adjust threshold in search_memory() if needed.")

# Test with an unrelated question — should not match
unrelated = "What are the applications of computer vision in medical imaging?"
match = search_memory(unrelated)
if not match:
    print("\nUnrelated question correctly returned no memory match.")

In [ ]:
app_source = open("/content/app.py").read() if os.path.exists("/content/app.py") else ""
print(f"app.py present: {bool(app_source)} ({len(app_source)} chars)")

# If app.py is not on disk, write it here.
# The full app.py file is maintained separately in this repository.
# See app.py in the repo root.

In [ ]:
import subprocess
import time
from pyngrok import ngrok
from google.colab import userdata

NGROK_TOKEN = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Reinstall packages in case the kernel restarted
subprocess.run([
    "pip", "install", "-q",
    "groq", "langchain", "langchain-groq", "langgraph", "langchain-community",
    "qdrant-client", "sentence-transformers", "arxiv", "duckduckgo-search",
    "pymupdf", "streamlit", "pyngrok"
], check=True)

# Stop any running Streamlit instance
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(3)

process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ],
    env=os.environ.copy()
)

time.sleep(8)
public_url = ngrok.connect(8501)
print(f"App running at: {public_url}")
print("The URL remains active while this cell is running.")